In [86]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [88]:
device = "cpu"

In [90]:
class FlightDataset(Dataset):
    def __init__(self, sequences_cat, sequences_cont, target_cat, target_cont, labels):
        self.sequences_cat = torch.tensor(sequences_cat,  dtype=torch.short)
        self.sequences_cont = torch.tensor(sequences_cont, dtype=torch.float32)
        self.target_cat  = torch.tensor(target_cat, dtype=torch.short)
        self.target_cont = torch.tensor(target_cont, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            self.sequences_cat[idx],
            self.sequences_cont[idx],
            self.target_cat[idx],
            self.target_cont[idx],
            self.labels[idx]
        )

In [96]:
class FlightDelayRNN(nn.Module):
    def __init__(
        self,
        n_origins,    
        n_dests,      
        n_cont_features,
        n_target_cont_features,
        embed_dim=16, 
        hidden_size =24
    ):
        super(FlightDelayRNN, self).__init__()

        self.origin_embed  = nn.Embedding(n_origins,  embed_dim)
        self.dest_embed    = nn.Embedding(n_dests,    embed_dim)
        self.carrier_embed = nn.Embedding(12, embed_dim) #12 carriers
        self.daysofweek_embed = nn.Embedding(7, embed_dim) #7 days of week

        self.cont_bn = nn.BatchNorm1d(n_cont_features) #batch normalization

        lstm_input_size = (embed_dim * 4) + n_cont_features
        
        self.lstm = nn.LSTM(
            input_size=lstm_input_size,
            hidden_size=hidden_size,
            batch_first=True,
        )

        target_feature_size = (embed_dim * 4) + n_target_cont_features
        self.dense = nn.Linear(hidden_size+target_feature_size,1)

    def forward(self, x_cat, x_cont, target_cat, target_cont):
        #embedding categorical features
        origin_emb = self.origin_embed(x_cat[:, :, 0])   # (batch, seq_len, embed_dim)
        dest_emb = self.dest_embed(x_cat[:, :, 1])     # (batch, seq_len, embed_dim)
        carrier_emb = self.carrier_embed(x_cat[:, :, 2])  # (batch, seq_len, embed_dim)
        daysofweek_emb = self.daysofweek_embed(x_cat[:,:,3]) # (batch, seq_len, embed_dim)
        
        x = torch.cat([origin_emb, dest_emb, carrier_emb, daysofweek_emb, x_cont_norm], dim=-1)

        output, (h_n, c_n) = self.lstm(x)
        final_hidden = h_n[-1]

        t_origin_emb = self.origin_embed(target_cat[:, 0]) 
        t_dest_emb = self.dest_embed(target_cat[:, 1])
        t_carrier_emb = self.carrier_embed(target_cat[:, 2])
        t_daysofweek_emb = self.daysofweek_emb(target_cat[:,3])
        
        target = torch.cat( [t_origin_emb, t_dest_emb, t_carrier_emb, t_daysofweek_emb, target_cont], dim=-1)

        observation = torch.cat([final_hidden, target], dim=-1)
        return self.dense(combined)

In [94]:
def train_model(model, train_loader, val_loader):  
    criterion = torch.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    best_val_loss = float('inf')

    #5 epochs for now
    for epoch in range(5):
        model.train()
        train_losses = []

        #training
        for x_cat, x_cont, x_target_cat, x_target_cont, labels in train_loader:
            x_cat = x_cat.to(device)
            x_cont = x_cont.to(device)
            x_target_cat = x_target_cat.to(device)
            x_target_cont = x_target_cont.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            predictions = model(x_cat, x_cont, x_target_cat, x_target_cont)
            loss = criterion(predictions, labels)
            loss.backward()

            # Gradient clipping — important for RNNs
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            train_losses.append(loss.item())

        
        model.eval()
        val_losses = []

        #validation
        with torch.no_grad():
            for x_cat, x_cont, x_target_cat, x_target_cont, labels in val_loader:
                x_cat = x_cat.to(device)
                x_cont = x_cont.to(device)
                x_target_cat = x_target_cat.to(device)
                x_target_cont = x_target_cont.to(device)
                labels = labels.to(device)

                predictions = model(x_cat, x_cont, x_target_cat, x_target_cont)
                loss = criterion(predictions, labels)
                val_losses.append(loss.item())

        avg_train_loss = np.mean(train_losses)
        avg_val_loss = np.mean(val_losses)
        scheduler.step(avg_val_loss)

        # save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), 'best_model.pt')

        print(f'Epoch {epoch+1:3d} | '
              f'Train Loss: {avg_train_loss:.4f} | '
              f'Val Loss: {avg_val_loss:.4f}')

    print('Training complete. Best model saved.')

In [ ]:
df = pd.read_csv('C:\\Users\\natha\\Documents\\combined_weather_dec2020\\2020_12_with_weather.csv')

In [82]:
cat_features = ['DayOfWeek', 'Dest', 'Origin', 'Reporting_Airline']
cont_features = ['delta_t', 'Year', 'CRSDepTime', 'CRSArrTime', 'dep_mx2t', 'dep_mn2t', 'dep_mxtpr', 'dep_fg10', 'dep_u10', 'dep_v10', 
                 'arr_mx2t', 'arr_mn2t', 'arr_mxtpr', 'arr_fg10', 'arr_u10', 'arr_v10', 'DepDelay']
target_cat_features = ['DayOfWeek', 'Dest', 'Origin', 'Reporting_Airline']
target_cont_features = ['delta_t', 'Year', 'CRSDepTime', 'CRSArrTime', 'dep_mx2t', 'dep_mn2t', 'dep_mxtpr', 'dep_fg10', 'dep_u10', 'dep_v10', 
                 'arr_mx2t', 'arr_mn2t', 'arr_mxtpr', 'arr_fg10', 'arr_u10', 'arr_v10']

In [70]:
import datetime
import airportsdata
import zoneinfo

In [64]:
iata_airports = airportsdata.load('IATA')

In [76]:
# takes in a date and military time at a certain airport and returns the UTC time
def convert_date(year, month, day, hhmm, airport):
    dt = datetime.strptime(str(hhmm).zfill(4), '%H%M')
    dt = dt.replace(year = year, month=month, day = day, tzinfo = zoneinfo.ZoneInfo(iata_airports[airport]['tz']))
    return dt.astimezone(timezone.utc)

In [84]:
def build_sequences(data, sequence_length):
    data['UTC_CRSDepTime'] = data.apply(lambda row: convert_date(row['Year'], row['Month'], row['Day'], row['CRSDepTime'], row['Origin']))
    
    X_cat_list = []
    X_cont_list = []
    X_target_cat_list = []
    X_target_cont_list = []
    y_list = []

    for tail, group in df.groupby('Tail_Number'):
        group = group.sort_values('CRSDepTime').reset_index(drop=True)

        group['delta_t'] = group['UTC_CRSDepTime'].diff().fillna(pd.Timedelta(0)).total_seconds() / 3600

        cat_values = group[cat_features].to_numpy(dtype=str)
        cont_values = group[cont_features].to_numpy(dtype=float)
        labels = group['DepDelay'].to_numpy() 

        for i in range(len(group) - sequence_length):
            X_cat_list.append(cat_values[i : i + sequence_length])
            X_cont_list.append(cont_values[i : i + sequence_length])
            X_target_cat_list.append(target_cat_values[i+sequence_length]) 
            X_target_cont_list.append(target_cont_values[i+sequence_length])

            y_list.append(labels[i + sequence_length])

    X_cat  = np.array(X_cat_list)
    X_cont = np.array(X_cont_list)
    X_target_cat = np.array(X_target_cat_list)
    X_target_cont = np.array(X_target_cont_list)
    y = np.array(y_list)

    return X_cat, X_cont, X_target_cat, X_target_cont, y


In [ ]:
df['origin_enc'] = OneHotEncoder(drop='first', handle_unknown='ignore').fit_transform(df['Origin'])
df['dest_enc'] = OneHotEncoder(drop='first', handle_unknown='ignore').fit_transform(df['Dest'])
df['carrier_enc'] = OneHotEncoder(drop='first', handle_unknown='ignore').fit_transform(df['Reporting_Airline'])
df['daysofweek_enc'] = OneHotEncoder(drop='first').fit_transform(df['DayOfWeek'])

X_cat, X_cont, X_target_cat, X_target_cont, y = build_sequences(df, sequence_length=5)

train_end = int(len(y) * 0.80)

X_cat_train,  X_cat_val= X_cat[:train_end],  X_cat[train_end:]
X_cont_train, X_cont_val = X_cont[:train_end], X_cont[train_end:]
X_target_cat_train,  X_target_cat_val = X_cat[:train_end],  X_cat[train_end:]
X_target_cont_train, X_target_cont_val = X_cont[:train_end], X_cont[train_end]
y_train, y_val = y[:train_end], y[train_end:]

train_dataset = FlightDataset(X_cat_train, X_cont_train, X_target_cat_train, X_target_cont_train, y_train)
val_dataset = FlightDataset(X_cat_val,   X_cont_val, X_target_cat_val, X_target_cont_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader = DataLoader(val_dataset,   batch_size=256, shuffle=False)

model = FlightDelayRNN(
    n_origins = df['origin_enc'].nunique(),
    n_dests = df['dest_enc'].nunique(),
    n_cont_features = len(cont_features)
    n_target_cont_features = len(target_cont_features)
).to(device)

train_model(model, train_loader, val_loader)